In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import serial
import time

class KalmanFilter3D:
    def __init__(self, dt, process_noise_std, measurement_noise_std):
        # Time step
        self.dt = dt
        
        # State vector [angular_velocity_x, angular_velocity_y, angular_velocity_z, acceleration_x, acceleration_y, acceleration_z]
        self.x = np.zeros((6, 1))
        
        # State covariance matrix
        self.P = np.eye(6)
        
        # Process noise covariance matrix
        self.Q = np.eye(6) * process_noise_std**2
        
        # Measurement noise covariance matrix
        self.R = np.eye(6) * measurement_noise_std**2
        
        # Measurement matrix
        self.H = np.eye(6)
        
        # State transition matrix
        self.A = np.eye(6)
        self.A[0, 3] = self.dt
        self.A[1, 4] = self.dt
        self.A[2, 5] = self.dt

    def predict(self): #prediction dtep
        # Predict the next state
        self.x = self.A @ self.x
        # Predict the next covariance
        self.P = self.A @ self.P @ self.A.T + self.Q

    def update(self, z): #update step
        # Kalman gain
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        
        # Update the state with the new measurement
        y = z.reshape(6, 1) - self.H @ self.x
        self.x = self.x + K @ y
        
        # Update the covariance matrix
        I = np.eye(6)
        self.P = (I - K @ self.H) @ self.P

    def get_state(self):
        return self.x.flatten()


# Serial port configuration
port = 'COM3'  # Replace with your Arduino's serial port
baudrate = 9600
timeout = 1

# Open serial port
ser = serial.Serial(port, baudrate, timeout=timeout)

# Variables for handling timestamps
previous_millis = int(round(time.time() * 1000))  # Current time in milliseconds

previous_velocity=np.zero(6)

# Read and process data
try:
    while True:
        current_millis = int(round(time.time() * 1000))
        elapsed_millis = current_millis - previous_millis
        
        if elapsed_millis >= 60:  # Example delay of 100 milliseconds
            previous_millis = current_millis
            
            line = ser.readline().decode('utf-8').strip()
            
            if line:
                # Split the line into timestamp and sensor data
                data = line.split(',')
                timestamp = int(data[0])
                accel_x = int(data[1])
                accel_y = int(data[2])
                accel_z = int(data[3])
                gyro_x = int(data[4])
                gyro_y = int(data[5])
                gyro_z = int(data[6])
                
                process_noise_std = 0.1   #to be set according to imu
                measurement_noise_std = 0.02   #to be set according to imu

                # Process your data here
                kf = KalmanFilter3D(elapsed_time, process_noise_std, measurement_noise_std)
                measurements=np.array([accel_x, accel_y, accel_z, gyro_x, gyro_y, gyro_z]).reshape((6, 1))
                filtered_accel = []

                for measurement in measurements:
                  kf.predict()
                  kf.update(np.array(measurement))
                  filtered_accel.append(kf.get_state())
                  print("State estimate:", kf.get_state().flatten())
                # print(f"Timestamp: {timestamp}, Accelerometer: ({ax}, {ay}, {az}), Gyroscope: ({gx}, {gy}, {gz})")

                filtered_accel = np.array(estimated_states)
                measurements = np.array(measurements)
                
                # Calculate velocity by integrating acceleration
                current_velocity = previous_velocity + filtered_accel * dt

                # Calculate displacement by integrating velocity
                current_displacement = previous_displacement + current_velocity * dt
                
                print(current_displacement)

                plt.figure()
                plt.subplot(311)
                plt.plot(timestamps, current_displacement)
                plt.labelx('timestamps')
                plt.labely('Displacement')
                plt.title('Displacement')
                plt.show()

                # Update previous values for next iteration
                previous_velocity = current_velocity
                previous_displacement = current_displacement
        
except KeyboardInterrupt:
    print("Serial communication stopped by user.")
    ser.close()


SerialException: could not open port 'COM3': FileNotFoundError(2, 'The system cannot find the file specified.', None, 2)